# Notebook 08 — Model Comparison 


## 0. Imports and configuration

In [1]:
from pathlib import Path
import json
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

EYE_DATASET_DIR = PROJECT_ROOT / "eye_dataset"
MOUTH_DATASET_DIR = PROJECT_ROOT / "mouth_dataset"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

EYE_CLASSES = ["Closed", "Open"]
MOUTH_CLASSES = ["no_yawn", "yawn"]

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)


TensorFlow: 2.21.0
Project root: d:\driver-drowsiness-detection


## 1. Locate the saved evaluation summaries



In [2]:
expected_files = [
    OUTPUT_DIR / "eye_custom_cnn_v2_metrics.json",
    OUTPUT_DIR / "eye_mobilenetv2_metrics.json",
    OUTPUT_DIR / "mouth_model_comparison_summary.json",
    OUTPUT_DIR / "mouth_model_comparison.csv"
]

for path in expected_files:
    print("FOUND" if path.exists() else "MISSING", "->", path)

missing = [str(p) for p in expected_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))


FOUND -> d:\driver-drowsiness-detection\outputs\eye_custom_cnn_v2_metrics.json
FOUND -> d:\driver-drowsiness-detection\outputs\eye_mobilenetv2_metrics.json
FOUND -> d:\driver-drowsiness-detection\outputs\mouth_model_comparison_summary.json
FOUND -> d:\driver-drowsiness-detection\outputs\mouth_model_comparison.csv


## 2. Load saved model metrics

In [3]:
def load_json(path):
    if not path.exists():
        raise FileNotFoundError(f"Required metrics file not found: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

eye_custom_v2_metrics = load_json(OUTPUT_DIR / "eye_custom_cnn_v2_metrics.json")
eye_mobilenet_metrics = load_json(OUTPUT_DIR / "eye_mobilenetv2_metrics.json")
mouth_summary = load_json(OUTPUT_DIR / "mouth_model_comparison_summary.json")
mouth_comparison = pd.read_csv(OUTPUT_DIR / "mouth_model_comparison.csv")

print("Eye Custom CNN v2:")
print(json.dumps(eye_custom_v2_metrics, indent=2))
print("\nEye MobileNetV2:")
print(json.dumps(eye_mobilenet_metrics, indent=2))
print("\nMouth model comparison:")
display(mouth_comparison)


Eye Custom CNN v2:
{
  "model_name": "eye_state_custom_cnn_v2",
  "input_size": [
    224,
    224
  ],
  "num_classes": 2,
  "class_names": [
    "Closed",
    "Open"
  ],
  "batch_size": 32,
  "epochs_requested": 40,
  "epochs_completed": 40,
  "optimizer": "Adam",
  "initial_learning_rate": 0.0001,
  "label_smoothing": 0.0,
  "augmentation": {
    "horizontal_flip": true,
    "rotation_factor": 0.05,
    "zoom_factor": 0.1,
    "contrast_factor": 0.1
  },
  "dropout": 0.4,
  "class_weights": {
    "0": 0.9865384615384616,
    "1": 1.0138339920948616
  },
  "test_metrics": {
    "accuracy": 0.9626168224299065,
    "macro_precision": 0.962438853948288,
    "macro_recall": 0.9634103641456582,
    "macro_f1": 0.9625874125874125,
    "weighted_precision": 0.9633286963563812,
    "weighted_recall": 0.9626168224299065,
    "weighted_f1": 0.9626364289915692
  },
  "confusion_matrix": [
    [
      100,
      2
    ],
    [
      6,
      106
    ]
  ],
  "confidence": {
    "mean": 0.943566

,Model,Accuracy,Precision,Recall,F1,Yawn_TP,Yawn_FN
0,Mouth Custom CNN,0.533937,0.0,0.000000,0.000000,0,103
1,Mouth MobileNetV2,0.977376,1.0,0.951456,0.975124,98,5


## 3. Build a unified model comparison table


In [4]:
def eye_metric(metrics, name):
    # v2 stores metrics under test_metrics; the older MobileNet file stores them at root.
    if "test_metrics" in metrics:
        return float(metrics["test_metrics"][name])
    return float(metrics[name])

comparison_rows = [
    {
        "Model": "Eye Custom CNN v2", "Task": "Eye",
        "Accuracy": eye_metric(eye_custom_v2_metrics, "accuracy"),
        "Precision": eye_metric(eye_custom_v2_metrics, "weighted_precision"),
        "Recall": eye_metric(eye_custom_v2_metrics, "weighted_recall"),
        "F1": eye_metric(eye_custom_v2_metrics, "weighted_f1"),
        "Test_Size": int(sum(eye_custom_v2_metrics.get("confusion_matrix", [[], []])[0])) + int(sum(eye_custom_v2_metrics.get("confusion_matrix", [[], []])[1]))
    },
    {
        "Model": "Eye MobileNetV2", "Task": "Eye",
        "Accuracy": eye_mobilenet_metrics["test_accuracy"],
        "Precision": eye_mobilenet_metrics["test_precision_weighted"],
        "Recall": eye_mobilenet_metrics["test_recall_weighted"],
        "F1": eye_mobilenet_metrics["test_f1_weighted"],
        "Test_Size": None
    }
]

for _, row in mouth_comparison.iterrows():
    comparison_rows.append({
        "Model": row["Model"], "Task": "Mouth",
        "Accuracy": row["Accuracy"], "Precision": row["Precision"],
        "Recall": row["Recall"], "F1": row["F1"], "Test_Size": None
    })

unified_comparison = pd.DataFrame(comparison_rows)
display(unified_comparison.style.format({"Accuracy":"{:.4f}","Precision":"{:.4f}","Recall":"{:.4f}","F1":"{:.4f}"}))


,Model,Task,Accuracy,Precision,Recall,F1,Test_Size
0,Eye Custom CNN v2,Eye,0.9626,0.9633,0.9626,0.9626,214.000000
1,Eye MobileNetV2,Eye,1.0000,1.0000,1.0000,1.0000,nan
2,Mouth Custom CNN,Mouth,0.5339,0.0000,0.0000,0.0000,nan
3,Mouth MobileNetV2,Mouth,0.9774,1.0000,0.9515,0.9751,nan


## 4. Select the final model for each task



In [5]:
def select_eye_model(metrics_a, metrics_b):
    rows = [("Eye Custom CNN v2", metrics_a), ("Eye MobileNetV2", metrics_b)]
    selected_name, selected_metrics = max(
        rows,
        key=lambda item: (
            eye_metric(item[1], "weighted_f1") if item[0] == "Eye Custom CNN v2" else item[1]["test_f1_weighted"],
            eye_metric(item[1], "weighted_recall") if item[0] == "Eye Custom CNN v2" else item[1]["test_recall_weighted"],
            eye_metric(item[1], "accuracy") if item[0] == "Eye Custom CNN v2" else item[1]["test_accuracy"]
        )
    )
    return selected_name, selected_metrics

eye_selected_name, eye_selected_metrics = select_eye_model(eye_custom_v2_metrics, eye_mobilenet_metrics)
print("Selected eye model:", eye_selected_name)

if eye_selected_name == "Eye Custom CNN v2":
    print("F1:", eye_metric(eye_selected_metrics, "weighted_f1"))
    print("Recall:", eye_metric(eye_selected_metrics, "weighted_recall"))
    print("Accuracy:", eye_metric(eye_selected_metrics, "accuracy"))
else:
    print("F1:", eye_selected_metrics["test_f1_weighted"])
    print("Recall:", eye_selected_metrics["test_recall_weighted"])
    print("Accuracy:", eye_selected_metrics["test_accuracy"])

mouth_selection_rows = []
for _, row in mouth_comparison.iterrows():
    model_name = row["Model"]
    cm_key = "custom_cnn" if "Custom CNN" in model_name else "mobilenetv2"
    cm = np.asarray(mouth_summary[cm_key]["confusion_matrix"])
    yawn_recall = cm[1,1] / cm[1].sum() if cm[1].sum() > 0 else 0.0
    mouth_selection_rows.append({"Model":model_name,"F1":float(row["F1"]),"Recall":float(row["Recall"]),"Accuracy":float(row["Accuracy"]),"Yawn_Recall":float(yawn_recall)})

mouth_selection_df = pd.DataFrame(mouth_selection_rows)
display(mouth_selection_df.style.format({"F1":"{:.4f}","Recall":"{:.4f}","Accuracy":"{:.4f}","Yawn_Recall":"{:.4f}"}))
mouth_selected_row = mouth_selection_df.sort_values(["Yawn_Recall","F1","Accuracy"], ascending=False).iloc[0]
mouth_selected_name = mouth_selected_row["Model"]
print("Selected mouth model:", mouth_selected_name)


Selected eye model: Eye MobileNetV2
F1: 1.0
Recall: 1.0
Accuracy: 1.0


,Model,F1,Recall,Accuracy,Yawn_Recall
0,Mouth Custom CNN,0.0000,0.0000,0.5339,0.0000
1,Mouth MobileNetV2,0.9751,0.9515,0.9774,0.9515


Selected mouth model: Mouth MobileNetV2


## 5. Load the selected models


In [6]:
eye_model_paths = {
    "Eye Custom CNN v2": MODEL_DIR / "eye_custom_cnn_v2_best.keras",
    "Eye MobileNetV2": MODEL_DIR / "eye_mobilenetv2_best.keras"
}
mouth_model_paths = {
    "Mouth Custom CNN": MODEL_DIR / "mouth_custom_cnn_best.keras",
    "Mouth MobileNetV2": MODEL_DIR / "mouth_mobilenetv2_best.keras"
}

selected_eye_path = eye_model_paths[eye_selected_name]
selected_mouth_path = mouth_model_paths[mouth_selected_name]

for path in [selected_eye_path, selected_mouth_path]:
    if not path.exists():
        raise FileNotFoundError(f"Selected model not found: {path}")

selected_eye_model = keras.models.load_model(selected_eye_path)
selected_mouth_model = keras.models.load_model(selected_mouth_path)
print("Eye model:", selected_eye_path)
print("Mouth model:", selected_mouth_path)


Eye model: d:\driver-drowsiness-detection\models\eye_mobilenetv2_best.keras
Mouth model: d:\driver-drowsiness-detection\models\mouth_mobilenetv2_best.keras


## 6. Inspect confidence characteristics


In [7]:
CONFIDENCE_THRESHOLD = 0.75
print("Initial fusion confidence threshold:", CONFIDENCE_THRESHOLD)

if eye_selected_name == "Eye Custom CNN v2":
    print("Eye mean confidence:", eye_selected_metrics.get("confidence", {}).get("mean"))
    print("Eye median confidence:", eye_selected_metrics.get("confidence", {}).get("median"))
else:
    print("Eye mean confidence:", eye_selected_metrics.get("mean_prediction_confidence"))
    print("Eye median confidence:", eye_selected_metrics.get("median_prediction_confidence"))

mouth_selected_key = "custom_cnn" if "Custom CNN" in mouth_selected_name else "mobilenetv2"
print("\nSelected mouth model:", mouth_selected_name)
print("Mouth confusion matrix:")
print(np.asarray(mouth_summary[mouth_selected_key]["confusion_matrix"]))


Initial fusion confidence threshold: 0.75
Eye mean confidence: 0.999843
Eye median confidence: 1.0

Selected mouth model: Mouth MobileNetV2
Mouth confusion matrix:
[[118   0]
 [  5  98]]


## 7. Define confidence-aware prediction helpers


In [8]:
def binary_prediction_from_output(output, class_names, threshold=CONFIDENCE_THRESHOLD):
    probs = np.asarray(output).reshape(-1)
    if probs.size == 1:
        p_class1 = float(probs[0])
    elif probs.size == 2:
        p_class1 = float(probs[1])
    else:
        raise ValueError(f"Expected 1 or 2 output probabilities, got shape {probs.shape}")

    if p_class1 >= 0.50:
        state = class_names[1]
        confidence = p_class1
    else:
        state = class_names[0]
        confidence = 1.0 - p_class1

    return {
        "state": state,
        "confidence": float(confidence),
        "probability_class_1": p_class1,
        "confident": bool(confidence >= threshold)
    }

def predict_eye(image, threshold=CONFIDENCE_THRESHOLD):
    output = selected_eye_model.predict(np.expand_dims(image, axis=0), verbose=0)[0]
    return binary_prediction_from_output(output, EYE_CLASSES, threshold)

def predict_mouth(image, threshold=CONFIDENCE_THRESHOLD):
    output = selected_mouth_model.predict(np.expand_dims(image, axis=0), verbose=0)[0]
    return binary_prediction_from_output(output, MOUTH_CLASSES, threshold)

print("✓ Confidence-aware prediction helpers ready.")


✓ Confidence-aware prediction helpers ready.


# 8. Define the final decision-fusion rule


In [9]:
def fuse_states(
    eye_result,
    mouth_result,
    threshold=CONFIDENCE_THRESHOLD
):
    eye_state = eye_result["state"]
    eye_conf = eye_result["confidence"]

    mouth_state = mouth_result["state"]
    mouth_conf = mouth_result["confidence"]

    eye_confident = eye_conf >= threshold
    mouth_confident = mouth_conf >= threshold

    # 1. Severe fatigue has highest priority.
    if (
        eye_state == "Closed"
        and eye_confident
    ):
        return {
            "fatigue_state": "Severe Fatigue",
            "reason": "Confident closed-eye detection",
            "confidence": eye_conf,
            "eye_state": eye_state,
            "eye_confidence": eye_conf,
            "mouth_state": mouth_state,
            "mouth_confidence": mouth_conf
        }

    # 2. Mild fatigue requires a confident yawn
    # and no confident closed-eye override.
    if (
        mouth_state == "yawn"
        and mouth_confident
    ):
        return {
            "fatigue_state": "Mild Fatigue",
            "reason": "Confident yawn without confident closed-eye override",
            "confidence": mouth_conf,
            "eye_state": eye_state,
            "eye_confidence": eye_conf,
            "mouth_state": mouth_state,
            "mouth_confidence": mouth_conf
        }

    # 3. Alert requires both components to be confident.
    if (
        eye_state == "Open"
        and eye_confident
        and mouth_state == "no_yawn"
        and mouth_confident
    ):
        return {
            "fatigue_state": "Alert",
            "reason": "Confident open-eye and no-yawn detection",
            "confidence": min(
                eye_conf,
                mouth_conf
            ),
            "eye_state": eye_state,
            "eye_confidence": eye_conf,
            "mouth_state": mouth_state,
            "mouth_confidence": mouth_conf
        }

    # 4. Ambiguous or insufficient evidence.
    return {
        "fatigue_state": "Uncertain",
        "reason": "Insufficient confidence or conflicting evidence",
        "confidence": min(
            eye_conf,
            mouth_conf
        ),
        "eye_state": eye_state,
        "eye_confidence": eye_conf,
        "mouth_state": mouth_state,
        "mouth_confidence": mouth_conf
    }


print("✓ Fusion rule created.")


✓ Fusion rule created.


## 9. Unit-test every important state combination


In [10]:
def mock_result(state, confidence):
    probability = (
        confidence
        if state in ["Open", "yawn"]
        else 1.0 - confidence
    )

    return {
        "state": state,
        "confidence": confidence,
        "probability_class_1": probability,
        "confident": confidence >= CONFIDENCE_THRESHOLD
    }


test_cases = [
    {
        "name": "Open + no_yawn",
        "eye": ("Open", 0.95),
        "mouth": ("no_yawn", 0.94),
        "expected": "Alert"
    },
    {
        "name": "Open + yawn",
        "eye": ("Open", 0.95),
        "mouth": ("yawn", 0.93),
        "expected": "Mild Fatigue"
    },
    {
        "name": "Closed + no_yawn",
        "eye": ("Closed", 0.96),
        "mouth": ("no_yawn", 0.95),
        "expected": "Severe Fatigue"
    },
    {
        "name": "Closed + yawn",
        "eye": ("Closed", 0.97),
        "mouth": ("yawn", 0.96),
        "expected": "Severe Fatigue"
    },
    {
        "name": "Uncertain eye + no_yawn",
        "eye": ("Open", 0.60),
        "mouth": ("no_yawn", 0.95),
        "expected": "Uncertain"
    },
    {
        "name": "Open + uncertain mouth",
        "eye": ("Open", 0.95),
        "mouth": ("no_yawn", 0.60),
        "expected": "Uncertain"
    },
    {
        "name": "Low confidence closed eye",
        "eye": ("Closed", 0.60),
        "mouth": ("no_yawn", 0.95),
        "expected": "Uncertain"
    }
]

unit_results = []

for case in test_cases:

    result = fuse_states(
        mock_result(*case["eye"]),
        mock_result(*case["mouth"])
    )

    passed = (
        result["fatigue_state"]
        == case["expected"]
    )

    unit_results.append({
        "Test": case["name"],
        "Expected": case["expected"],
        "Actual": result["fatigue_state"],
        "Passed": passed
    })

unit_df = pd.DataFrame(unit_results)

display(unit_df)

if not unit_df["Passed"].all():
    raise AssertionError(
        "One or more fusion unit tests failed."
    )

print(
    "✓ All fusion unit tests passed."
)


,Test,Expected,Actual,Passed
0,Open + no_yawn,Alert,Alert,True
1,Open + yawn,Mild Fatigue,Mild Fatigue,True
2,Closed + no_yawn,Severe Fatigue,Severe Fatigue,True
3,Closed + yawn,Severe Fatigue,Severe Fatigue,True
4,Uncertain eye + no_yawn,Uncertain,Uncertain,True
5,Open + uncertain mouth,Uncertain,Uncertain,True
6,Low confidence closed eye,Uncertain,Uncertain,True


✓ All fusion unit tests passed.


## 10. Demonstrate the complete state-transition matrix


In [11]:
states_matrix = []

for eye_state in ["Open", "Closed"]:
    for mouth_state in ["no_yawn", "yawn"]:

        result = fuse_states(
            mock_result(
                eye_state,
                0.95
            ),
            mock_result(
                mouth_state,
                0.95
            )
        )

        states_matrix.append({
            "Eye State": eye_state,
            "Mouth State": mouth_state,
            "Fusion Output": result["fatigue_state"],
            "Reason": result["reason"]
        })

state_matrix_df = pd.DataFrame(
    states_matrix
)

display(state_matrix_df)


,Eye State,Mouth State,Fusion Output,Reason
0,Open,no_yawn,Alert,Confident open-eye and no-yawn detection
1,Open,yawn,Mild Fatigue,Confident yawn without confident closed-eye ov...
2,Closed,no_yawn,Severe Fatigue,Confident closed-eye detection
3,Closed,yawn,Severe Fatigue,Confident closed-eye detection


## 12. Save the final fusion configuration

This configuration will be consumed by the final Streamlit application.

The confidence threshold is intentionally stored as a configuration value rather than hidden inside application code.


In [13]:
fusion_config = {
    "eye_model": {"name": eye_selected_name, "path": str(selected_eye_path), "classes": EYE_CLASSES},
    "mouth_model": {"name": mouth_selected_name, "path": str(selected_mouth_path), "classes": MOUTH_CLASSES},
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "outputs": ["Alert", "Mild Fatigue", "Severe Fatigue", "Uncertain"],
    "decision_rules": {
        "closed_eye": "Severe Fatigue",
        "open_eye_yawn": "Mild Fatigue",
        "open_eye_no_yawn": "Alert",
        "insufficient_or_conflicting_confidence": "Uncertain"
    },
    "end_to_end_fusion_accuracy": None,
    "end_to_end_fusion_note": "Requires paired eye+mouth ground truth from the same frames."
}

fusion_config_path = OUTPUT_DIR / "final_fusion_config.json"
fusion_config_path.write_text(json.dumps(fusion_config, indent=2), encoding="utf-8")
print(json.dumps(fusion_config, indent=2))
print("\nSaved:", fusion_config_path)


{
  "eye_model": {
    "name": "Eye MobileNetV2",
    "path": "d:\\driver-drowsiness-detection\\models\\eye_mobilenetv2_best.keras",
    "classes": [
      "Closed",
      "Open"
    ]
  },
  "mouth_model": {
    "name": "Mouth MobileNetV2",
    "path": "d:\\driver-drowsiness-detection\\models\\mouth_mobilenetv2_best.keras",
    "classes": [
      "no_yawn",
      "yawn"
    ]
  },
  "confidence_threshold": 0.75,
  "outputs": [
    "Alert",
    "Mild Fatigue",
    "Severe Fatigue",
    "Uncertain"
  ],
  "decision_rules": {
    "closed_eye": "Severe Fatigue",
    "open_eye_yawn": "Mild Fatigue",
    "open_eye_no_yawn": "Alert",
    "insufficient_or_conflicting_confidence": "Uncertain"
  },
  "end_to_end_fusion_accuracy": null,
  "end_to_end_fusion_note": "Requires paired eye+mouth ground truth from the same frames."
}

Saved: d:\driver-drowsiness-detection\outputs\final_fusion_config.json


## 13. Save the comparison and fusion report

In [14]:
report = {
    "eye_model_selected": eye_selected_name,
    "mouth_model_selected": mouth_selected_name,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "unit_tests_passed": bool(unit_df["Passed"].all()),
    "end_to_end_fusion_accuracy": None,
    "reason_end_to_end_accuracy_missing": "No paired eye+mouth ground-truth dataset is available.",
    "models": unified_comparison.to_dict(orient="records")
}
report_path = OUTPUT_DIR / "model_comparison_fusion_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print("\nSaved:", report_path)


{
  "eye_model_selected": "Eye MobileNetV2",
  "mouth_model_selected": "Mouth MobileNetV2",
  "confidence_threshold": 0.75,
  "unit_tests_passed": true,
  "end_to_end_fusion_accuracy": null,
  "reason_end_to_end_accuracy_missing": "No paired eye+mouth ground-truth dataset is available.",
  "models": [
    {
      "Model": "Eye Custom CNN v2",
      "Task": "Eye",
      "Accuracy": 0.9626168224299065,
      "Precision": 0.9633286963563812,
      "Recall": 0.9626168224299065,
      "F1": 0.9626364289915692,
      "Test_Size": 214.0
    },
    {
      "Model": "Eye MobileNetV2",
      "Task": "Eye",
      "Accuracy": 1.0,
      "Precision": 1.0,
      "Recall": 1.0,
      "F1": 1.0,
      "Test_Size": NaN
    },
    {
      "Model": "Mouth Custom CNN",
      "Task": "Mouth",
      "Accuracy": 0.5339366515837104,
      "Precision": 0.0,
      "Recall": 0.0,
      "F1": 0.0,
      "Test_Size": NaN
    },
    {
      "Model": "Mouth MobileNetV2",
      "Task": "Mouth",
      "Accuracy": 0.97